# YOLOv8n fine-tuning - Mendeley waste dataset\n\nThis Colab notebook reproduces the repository fine-tuning flow for the four project classes: `paper`, `plastic`, `glass`, and `residual`.\n\nMendeley dataset page: https://data.mendeley.com/datasets/2x69gjbcz6/2\n\nDownload the dataset from Mendeley, upload the downloaded zip here, then run the cells in order.

In [ ]:
!nvidia-smi\n!pip install -q ultralytics pyyaml

## Get the project code\n\nIf the repository is public, set `REPO_URL` and run the clone command. If not, upload a zip of this repository to Colab and unzip it manually, then set `PROJECT_DIR` to that folder.

In [ ]:
from pathlib import Path\n\nREPO_URL = ''  # Optional: paste your GitHub repo URL here\nPROJECT_DIR = Path('/content/RAPF-RM')\n\nif REPO_URL and not PROJECT_DIR.exists():\n    !git clone {REPO_URL} {PROJECT_DIR}\n\nif not PROJECT_DIR.exists():\n    PROJECT_DIR = Path('/content')\n\nprint('Project directory:', PROJECT_DIR)

## Upload and extract Mendeley dataset\n\nUse the Mendeley **Download All** button from the dataset page. Upload the zip file when this cell asks for it.

In [ ]:
from google.colab import files\nimport zipfile\n\nuploaded = files.upload()\nzip_path = Path(next(iter(uploaded)))\nraw_dir = Path('/content/mendeley_raw')\nraw_dir.mkdir(parents=True, exist_ok=True)\n\nwith zipfile.ZipFile(zip_path, 'r') as archive:\n    archive.extractall(raw_dir)\n\nprint('Extracted to:', raw_dir)\nprint('Example files:', list(raw_dir.rglob('*'))[:10])

## Prepare 4-class YOLO dataset\n\nThe project mapping is:\n\n- `paper`: paper, cardboard\n- `plastic`: plastic, metal\n- `glass`: glass\n- `residual`: organic waste, battery waste, e-waste, cloth, other waste

In [ ]:
import sys\n\nsys.path.insert(0, str(PROJECT_DIR))\n\nfrom waste_robot_behavior.dataset.mendeley_yolo import prepare_mendeley_subset\n\nprepared_dir = PROJECT_DIR / 'data/mendeley_yolo_4bins'\nsummary = prepare_mendeley_subset(\n    source_dir=raw_dir,\n    output_dir=prepared_dir,\n    sample_size=600,\n    val_ratio=0.2,\n    seed=42,\n)\n\nprint(summary)\nprint((prepared_dir / 'data.yaml').read_text())

## Train YOLOv8n\n\nThe trained weights will be written under `runs/waste_yolov8/yolov8n_mendeley_4bins/weights/best.pt`.

In [ ]:
from ultralytics import YOLO\n\nmodel = YOLO('yolov8n.pt')\nresults = model.train(\n    data=str(prepared_dir / 'data.yaml'),\n    imgsz=640,\n    epochs=30,\n    batch=8,\n    project=str(PROJECT_DIR / 'runs/waste_yolov8'),\n    name='yolov8n_mendeley_4bins',\n)

## Download trained weights

In [ ]:
best_weights = PROJECT_DIR / 'runs/waste_yolov8/yolov8n_mendeley_4bins/weights/best.pt'\nprint('best.pt:', best_weights)\nfiles.download(str(best_weights))